# p.97

About the query on p.97, bottom, that shows joining the same tables multiple times. To me at first reading it wasn't obvious that specifying the two names in that order will filter rows with movies with those two actors - after all, what if they are not given in that order? However, having actually coded and checked the intermediate step output, I now understand that it does work. 

In [ ]:
/* This is the original query:
*/
WITH film AS (
  SELECT 1 film_id, 'BLOOD ARGONAUTS' title union all 
  select 2 film_id, 'TOWERS HURRICANE' title union all 
  select 3 film_id, 'WHATEVER OTHER TITLE' title
),

film_actor AS (
  SELECT 1 film_id, 1 actor_id union all 
  select 1 film_id, 2 actor_id union all 
  select 1 film_id, 3 actor_id union all

  select 2 film_id, 3 actor_id union all 
  select 2 film_id, 1 actor_id union all 
  select 2 film_id, 4 actor_id union all

  select 3 film_id, 4 actor_id
),

actor AS (
  SELECT 1 actor_id, 'CATE' first_name, 'MCQUEEN' last_name union all 
  select 2 actor_id, 'JOHN' first_name, 'WAYNE' last_name UNION ALL 
  SELECT 3 actor_id, 'CUBA' first_name, 'BIRCH' last_name union all 
  select 4 actor_id, 'JANE' first_name, 'DOE' last_name
)

SELECT
	f.title
FROM film f
INNER JOIN film_actor fa1 
	ON f.film_id = fa1.film_id
INNER JOIN actor a1
	ON fa1.actor_id = a1.actor_id
INNER JOIN film_actor fa2
	ON f.film_id = fa2.film_id
INNER JOIN actor a2
	ON fa2.actor_id = a2.actor_id
WHERE
        (a1.first_name = 'CATE' AND a1.last_name = 'MCQUEEN')
    AND (a2.first_name = 'CUBA' AND a2.last_name = 'BIRCH')

/* Result table is as expected in the book:

title
BLOOD ARGONAUTS
TOWERS HURRICANE
*/

In [ ]:
/* However, in the final statement
if we remove the WHERE clause 
and select more tables, 
we can actually see the result of all the joins - essentially, on the second join repetition the result kind of duplicate: 
*/
WITH film AS (
  SELECT 1 film_id, 'BLOOD ARGONAUTS' title union all 
  select 2 film_id, 'TOWERS HURRICANE' title union all 
  select 3 film_id, 'WHATEVER OTHER TITLE' title
),

film_actor AS (
  SELECT 1 film_id, 1 actor_id union all 
  select 1 film_id, 2 actor_id union all 
  select 1 film_id, 3 actor_id union all

  select 2 film_id, 3 actor_id union all 
  select 2 film_id, 1 actor_id union all 
  select 2 film_id, 4 actor_id union all

  select 3 film_id, 4 actor_id
),

actor AS (
  SELECT 1 actor_id, 'CATE' first_name, 'MCQUEEN' last_name union all 
  select 2 actor_id, 'JOHN' first_name, 'WAYNE' last_name UNION ALL 
  SELECT 3 actor_id, 'CUBA' first_name, 'BIRCH' last_name union all 
  select 4 actor_id, 'JANE' first_name, 'DOE' last_name
)

-- the section below is used to visualise the intermediate result
SELECT
	f.title,
    a1.first_name,
    a1.last_name,
    a2.first_name, 
    a2.last_name
FROM film f
INNER JOIN film_actor fa1 
	ON f.film_id = fa1.film_id
INNER JOIN actor a1
	ON fa1.actor_id = a1.actor_id
INNER JOIN film_actor fa2
	ON f.film_id = fa2.film_id
INNER JOIN actor a2
	ON fa2.actor_id = a2.actor_id


/*
title                  first_name   last_name   first_name   last_name
BLOOD ARGONAUTS        CATE         MCQUEEN     CATE         MCQUEEN
BLOOD ARGONAUTS        CATE         MCQUEEN     JOHN         WAYNE
BLOOD ARGONAUTS        CATE         MCQUEEN     CUBA         BIRCH
BLOOD ARGONAUTS        JOHN         WAYNE       CATE         MCQUEEN
BLOOD ARGONAUTS        JOHN         WAYNE       JOHN         WAYNE
BLOOD ARGONAUTS        JOHN         WAYNE       CUBA         BIRCH
BLOOD ARGONAUTS        CUBA         BIRCH       CATE         MCQUEEN
BLOOD ARGONAUTS        CUBA         BIRCH       JOHN         WAYNE
BLOOD ARGONAUTS        CUBA         BIRCH       CUBA         BIRCH
TOWERS HURRICANE       CUBA         BIRCH       CATE         MCQUEEN
TOWERS HURRICANE       CUBA         BIRCH       CUBA         BIRCH
TOWERS HURRICANE       CUBA         BIRCH       JANE         DOE
TOWERS HURRICANE       CATE         MCQUEEN     CATE         MCQUEEN
TOWERS HURRICANE       CATE         MCQUEEN     CUBA         BIRCH
TOWERS HURRICANE       CATE         MCQUEEN     JANE         DOE
TOWERS HURRICANE       JANE         DOE	        CATE         MCQUEEN
TOWERS HURRICANE       JANE         DOE	        CUBA         BIRCH
TOWERS HURRICANE       JANE         DOE	        JANE         DOE
WHATEVER OTHER TITLE   JANE         DOE         JANE         DOE
*/


# p. 100

For exercise 5-3, add specifications that:
- in output, include address1, address2, and city_id
- include unique combinations of addresses - e.g. "address1" - "address2" and "address2" - "address1" should be represented only once by one row

# p.170

There is an example of multicolumn subquery there that does work, but it's super not readable and the logic is kind of jumbled:

In [ ]:
/* Some toy data for minimal reproducibility:
*/
WITH film_actor AS (
	SELECT 120 actor_id, 63 film_id UNION ALL 
	SELECT 121 actor_id, 64 film_id UNION ALL
	SELECT 122 actor_id, 63 film_id
),

actor AS (
	SELECT 120 actor_id, 'MONROE' last_name UNION ALL 
	SELECT 121 actor_id, 'DOE' last_name UNION ALL 
	SELECT 122 actor_id, 'WHATEVER' last_name
),

film AS (
	SELECT 63 film_id, 'PG' rating UNION ALL 
	SELECT 64 film_id, 'PG' rating 
)

/* The query from the book. 
*/
SELECT
	actor_id, 
	film_id
FROM film_actor
WHERE (actor_id, film_id) IN (
	SELECT a.actor_id, f.film_id
	FROM actor a
	CROSS JOIN film f
	WHERE 
		a.last_name = 'MONROE'
		AND f.rating = 'PG'
)

-- actor_id|film_id|
-- --------+-------+
--      120|     63|

However, in my mind, a statement that utilises inner joins is much more readable:

In [ ]:
/* Some toy data for minimal reproducibility:
*/
WITH film_actor AS (
	SELECT 120 actor_id, 63 film_id UNION ALL 
	SELECT 121 actor_id, 64 film_id UNION ALL
	SELECT 122 actor_id, 63 film_id
),

actor AS (
	SELECT 120 actor_id, 'MONROE' last_name UNION ALL 
	SELECT 121 actor_id, 'DOE' last_name UNION ALL 
	SELECT 122 actor_id, 'WHATEVER' last_name
),

film AS (
	SELECT 63 film_id, 'PG' rating UNION ALL 
	SELECT 64 film_id, 'PG' rating 
)

SELECT
	fa.actor_id,
	fa.film_id
FROM film_actor fa
INNER JOIN actor a
	ON fa.actor_id = a.actor_id
INNER JOIN film f
	ON fa.film_id = f.film_id
WHERE
	a.last_name = 'MONROE'
	AND f.rating = 'PG'

-- actor_id|film_id|
-- --------+-------+
--      120|     63|

# p. 200

Exercise 10-3 - alternative solution that doesn't require adding 1 in the select statement (in my opinion, it's more readable):


In [ ]:
WITH temp1 AS (
	SELECT 0 col1 UNION ALL SELECT 10 UNION ALL SELECT 20 UNION ALL SELECT 30 UNION ALL SELECT 40 UNION ALL SELECT 50 UNION ALL SELECT 60 UNION ALL SELECT 70 UNION ALL SELECT 80 UNION ALL SELECT 90
),
temp2 AS (
	SELECT 1 col2 UNION ALL SELECT 2 UNION ALL SELECT 3 UNION ALL SELECT 4 UNION ALL SELECT 5 UNION ALL SELECT 6 UNION ALL SELECT 7 UNION ALL SELECT 8 UNION ALL SELECT 9 UNION ALL SELECT 10
),
cross_join AS (
	SELECT * FROM temp1
	CROSS JOIN temp2
)

SELECT col1 + col2
FROM cross_join

# p. 211

Exercise 11-2 - alternative solution: 


In [ ]:
SELECT 
	(SELECT COUNT(*) FROM film WHERE rating = 'G') "G",
	(SELECT COUNT(*) FROM film WHERE rating = 'PG') "PG",
	(SELECT COUNT(*) FROM film WHERE rating = 'PG-13') "PG_13",
	(SELECT COUNT(*) FROM film WHERE rating = 'R') "R",
	(SELECT COUNT(*) FROM film WHERE rating = 'NC-17') "NC_17"

# p. 250

Solution A (like in the book) - uses subqueries - is VERY HARD to READ and INTERPRET!

Solution B (mine) - with joins (left joins are used because there are no payments associated with Australia):

In [ ]:
SELECT
    co.country AS country,
    SUM(p.amount) AS tot_payments
FROM country co
LEFT JOIN city ci
    ON co.country_id = ci.country_id
LEFT JOIN address a
    ON ci.city_id = a.city_id
LEFT JOIN customer c
    ON a.address_id = c.address_id
LEFT JOIN payment p
    ON c.customer_id = p.customer_id
GROUP BY co.country
ORDER BY SUM(p.amount) DESC

# p. 265

An alternative solution to 15-2:

In [ ]:
SELECT
    CONCAT(
        'ALTER TABLE ',
        table_name,
        ' ADD INDEX ',
        index_name,
        ' (', 
        column_name,
        ');'
    ) AS index_creation_statement
FROM (
    SELECT
        table_name,
        index_name,
        column_name
    FROM information_schema.statistics
    WHERE
        table_schema = 'sakila'
        AND table_name = 'customer'
) AS a1;

# p. 279

What they are referring to as "rolling sum" would be more correctly termed as "cumulative sum":
- https://netflix.github.io/atlas-docs/asl/ref/rolling-sum/


So the query is correct, the only change would be in alias. 

# p. 284

For exercises 16-1 through 16-3, the answers use `RANK` function, but any other rank function could be used, since it seems that all tot_sales values are unique. 